<a href="https://colab.research.google.com/github/mcarden4-netizen/Homework-Intro-to-ML/blob/Homework/Homework5part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import torch
from google.colab import drive
drive.mount('/content/drive')

filetest = '/content/drive/My Drive/Housing.csv'


torch.set_printoptions(edgeitems=2)

data = pd.read_csv('/content/drive/My Drive/Housing.csv')
data.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [ ]:
specs1 = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']

X = data[specs1].values
Y = data['price'].values

In [ ]:
X = torch.tensor(X, dtype=torch.float32)
Y = torch.tensor(Y, dtype=torch.float32)

In [ ]:
Y = Y / 1e6

torch.manual_seed(50)

Ts = X.shape[0]
Vs = int(0.2 * Ts)

shuffled_indices = torch.randperm(Ts)

traindata = shuffled_indices[:-Vs]
valdata = shuffled_indices[-Vs:]

Xtrain = X[traindata]
Xval = X[valdata]

Ytrain = Y[traindata]
Yval = Y[valdata]

In [ ]:
Xtrainm = Xtrain.mean(dim=0)
Xtrainstd = Xtrain.std(dim=0)

train_X = (Xtrain - Xtrainm) / Xtrainstd
val_X = (Xval - Xtrainm) / Xtrainstd

In [ ]:
def model(X, params):
    return X @ params[:5] + params[5]

In [ ]:
def loss_fn(YP, Y):
    squared_diffs = (YP - Y) ** 2
    return squared_diffs.mean()

In [ ]:
def training_loop(n_epochs, learning_rate, params, Xtrain, Ytrain, Xval, Yval):

    for epoch in range(1, n_epochs + 1):

        if params.grad is not None:
            params.grad.zero_()

        YPtrain = model(Xtrain, params)

        train_loss = loss_fn(YPtrain, Ytrain)

        train_loss.backward()

        with torch.no_grad():
            params -= learning_rate * params.grad

        with torch.no_grad():

            YPval = model(Xval, params)

            val_loss = loss_fn(YPval, Yval)

        if epoch % 500 == 0:
            print('Epoch %d, Training Loss %f, Validation Loss %f' % (epoch, float(train_loss), float(val_loss)))

    return params

In [ ]:
learning_rates = [1e-1, 1e-2, 1e-3, 1e-4]

for learning_rate in learning_rates:

    print("\nLearning rate:", learning_rate)

    params = training_loop(n_epochs=5000, learning_rate=learning_rate, params=torch.tensor([1.0, 0.0, 0.0, 0.0, 0.0, 0.0], requires_grad=True), Xtrain=train_X, Ytrain=Ytrain, Xval=val_X, Yval=Yval)

    print("Final parameters:")
    print(params)

    # Calculate final validation loss
    with torch.no_grad():
        val_predictions = model(val_X, params)
        final_val_loss = loss_fn(val_predictions, Yval)

    print("Final validation loss:", float(final_val_loss))


Learning rate: 0.1
Epoch 500, Training Loss 1.505154, Validation Loss 1.642822
Epoch 1000, Training Loss 1.505154, Validation Loss 1.642822
Epoch 1500, Training Loss 1.505154, Validation Loss 1.642822
Epoch 2000, Training Loss 1.505154, Validation Loss 1.642822
Epoch 2500, Training Loss 1.505154, Validation Loss 1.642822
Epoch 3000, Training Loss 1.505154, Validation Loss 1.642822
Epoch 3500, Training Loss 1.505154, Validation Loss 1.642822
Epoch 4000, Training Loss 1.505154, Validation Loss 1.642822
Epoch 4500, Training Loss 1.505154, Validation Loss 1.642822
Epoch 5000, Training Loss 1.505154, Validation Loss 1.642822
Final parameters:
tensor([0.7454, 0.1235, 0.5845, 0.4730, 0.2929, 4.7924], requires_grad=True)
Final validation loss: 1.642822265625

Learning rate: 0.01
Epoch 500, Training Loss 1.505156, Validation Loss 1.642875
Epoch 1000, Training Loss 1.505154, Validation Loss 1.642820
Epoch 1500, Training Loss 1.505154, Validation Loss 1.642820
Epoch 2000, Training Loss 1.505154,